# Microplastic Detection — merged Roboflow datasets → YOLO (Kaggle)

This notebook pulls **every listed open-source microplastic dataset from Roboflow Universe**, merges them into one
clean dataset, trains a YOLO detector, evaluates it honestly, and exports it for the Raspberry Pi 4 on the AUV.

**Pipeline**
1. Download all datasets (latest version of each) in YOLO format
2. Map every dataset's class names onto one label scheme (`single` = one *microplastic* class, or `shape` = fiber / fragment / pellet / film)
3. **De-duplicate across datasets** — Universe datasets are often forks of each other and contain Roboflow-augmented copies. Duplicates are grouped (filename stem + flip/rotation-invariant perceptual hash, verified by pixel correlation) so a copy of a test image can never sit in the training set
4. Re-split 80 / 10 / 10 per source, with one image per duplicate group in val/test
5. Add underwater-style copies (colour cast, haze, blur, noise) to the **training set only**
6. Train YOLO11 at high resolution (particles are tiny)
7. Report test metrics overall, per class, **per source dataset**, and at Pi-friendly 640 px
8. Export ONNX + NCNN for the Raspberry Pi

### Kaggle setup (once)
- **Settings → Accelerator:** GPU T4 or P100
- **Settings → Internet:** On
- **Add-ons → Secrets:** add `ROBOFLOW_API_KEY` (Roboflow → Settings → API Keys) and attach it to this notebook
- For the full training run use **Save Version → Save & Run All (Commit)** so it keeps running when you close the tab

> **About the score:** the test-set number this produces is a *benchmark on public lab/microscope images*. It shows the model works;
> it is not the accuracy you will get underwater with the final camera. The per-source table and the optional hold-out run at the end are what show generalisation.

In [ ]:
%pip install -q -U ultralytics roboflow onnx onnxslim onnxruntime ncnn pnnx

## 1. Configuration

In [ ]:
import os

# Every Roboflow Universe microplastic dataset to merge: (workspace, project, version or None = latest).
# To add more: open the dataset on universe.roboflow.com — the URL is universe.roboflow.com/<workspace>/<project>
DATASETS = [
    ("microplasticstrial",              "microplastics-detection-zve60",       None),
    ("university-of-alabama-zwtwm",     "microplastic-detection",              None),
    ("mp-detection-and-classification", "microplastic-detect-classify-sqvmn",  None),
    ("space1-ixaxr",                    "microplastic-detection-vujai",        None),
    ("sciencefair2025workspace",        "microplastic-detection-a",            None),
    ("shilpipyxeda",                    "microplastic-detection-k3jdo",        None),
    ("euris-workspace",                 "microplastics-ibql8",                 None),
    ("euris-workspace",                 "microplastics-q2pp1",                 None),
    ("waste-vgig9",                     "detection_microplastics",             None),
    ("deadlock-95pmq",                  "microplastic-1-class",                None),
]

# ---- Labels ----
LABEL_MODE = "single"      # "single" -> one class 'microplastic' (uses ALL data, best score)
                           # "shape"  -> fiber / fragment / pellet / film (images with only generic 'microplastic' labels are skipped)
CLASS_OVERRIDES = {}       # force a mapping after you see the class table, e.g.
                           # {"mp": "microplastic", "frgament": "fragment", "debris": "__negative__", "weird": "__drop_image__"}
UNKNOWN_POLICY = "drop_image"   # what to do with images containing an unrecognised class: "drop_image" or "negative"

# ---- Data ----
SPLIT = (0.8, 0.1, 0.1)    # train / val / test (by unique image group, per source)
SEED = 42
PHASH_MAX_DIST = 12        # candidate duplicate if perceptual-hash distance <= this ...
DUP_CORR = 0.92            # ... AND high-pass pixel correlation >= this
UNDERWATER_AUG_FRACTION = 0.3   # extra underwater-styled copies of this fraction of TRAIN images (0 = off)
MAX_IMAGES_PER_SOURCE = None    # cap a huge source so it doesn't dominate (None = no cap)

# ---- Training ----
MODEL = "yolo11s.pt"       # yolo11n.pt = fastest on the Pi, yolo11m.pt = more accurate, slower
IMGSZ = 1024               # high resolution because particles are tiny
EPOCHS = 150
TIME_HOURS = 10.0          # training budget in hours — Kaggle kills any session at 12 h. When set, it REPLACES EPOCHS:
                           # YOLO trains as many epochs as fit in this time with a full LR schedule, then stops cleanly,
                           # leaving ~1 h for evaluation + export. Set None to use EPOCHS instead.
RESUME_FROM = None         # to continue an interrupted run, the path to its last.pt, e.g.
                           # "/kaggle/working/runs/train/weights/last.pt" or "/kaggle/input/<your-dataset>/last.pt"
PATIENCE = 30              # early stop if val mAP doesn't improve for this many epochs
BATCH = 16                 # lower to 8 if you hit CUDA out-of-memory
DEVICE = 0
WORKERS = 4

# ---- Paths ----
WORK = os.environ.get("MP_WORK", "/tmp/mp")          # scratch space (not saved, doesn't use the 20 GB output quota)
OUT  = os.environ.get("MP_OUT",  "/kaggle/working")  # saved outputs (weights, plots, exports)
RAW, MERGED = f"{WORK}/raw", f"{WORK}/merged"
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)

In [ ]:
import json, random, re, shutil, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_rows", 200); pd.set_option("display.width", 200)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
random.seed(SEED); np.random.seed(SEED)

## 2. Download every dataset

In [ ]:
from roboflow import Roboflow

def get_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception:
        key = os.environ.get("ROBOFLOW_API_KEY")
        if not key:
            raise RuntimeError("Add ROBOFLOW_API_KEY under Add-ons -> Secrets and attach it to this notebook.")
        return key

def latest_version(proj):
    try:
        nums = [int(str(v.version).split("/")[-1]) for v in proj.versions()]
        if nums: return max(nums)
    except Exception:
        pass
    try:
        nums = [int(str(v["id"]).split("/")[-1]) for v in proj.get_version_information()]
        if nums: return max(nums)
    except Exception:
        pass
    return None

rf = Roboflow(api_key=get_api_key())
SOURCES = []      # (source_id, local_dir)
for ws, pj, ver in DATASETS:
    sid, dest = f"{ws}/{pj}", f"{RAW}/{ws}__{pj}"
    if Path(dest, "data.yaml").exists():
        print(f"✓ cached   {sid}"); SOURCES.append((sid, dest)); continue
    try:
        proj = rf.workspace(ws).project(pj)
        v = ver or latest_version(proj)
        if v is None:
            print(f"✗ {sid}: no dataset versions published"); continue
        proj.version(v).download("yolov8", location=dest, overwrite=True)
        if Path(dest, "data.yaml").exists():
            SOURCES.append((sid, dest)); print(f"✓ {sid} (v{v})")
        else:
            print(f"✗ {sid}: download had no data.yaml (not a detection dataset?)")
    except Exception as e:
        print(f"✗ {sid}: {type(e).__name__}: {e}")

print(f"\n{len(SOURCES)}/{len(DATASETS)} datasets available")

## 3. Unify class names
Every dataset names its classes differently (`Fiber`, `fibre`, `Frgament`, `Beads`, `microplastic`, `dirt` …).
They are mapped by keyword. **Check the table below** — anything marked `__unknown__` is handled by `UNKNOWN_POLICY`;
fix it with `CLASS_OVERRIDES` in the config cell and re-run from there. Non-plastic classes (dirt, sand, organic …) are
dropped as labels but their images are kept — they teach the model what *not* to detect.

In [ ]:
SHAPE_KEYWORDS = {
    "fiber":    ["fiber", "fibre", "filament", "thread"],
    "fragment": ["fragment", "frgament", "fragmen", "frag", "foam"],
    "pellet":   ["pellet", "bead", "sphere", "nurdle", "granule"],
    "film":     ["film", "sheet", "membrane"],
}
SHAPE_CLASSES = list(SHAPE_KEYWORDS)
NEGATIVE_KEYWORDS = ["non plastic", "nonplastic", "not plastic", "no plastic", "dirt", "sand", "organic", "bubble",
                     "algae", "sediment", "biological", "natural", "soil", "plankton", "organism"]
GENERIC_EXACT = {"mp", "mps", "micro plastic", "microplastic", "microplastics"}

def _norm(s): return re.sub(r"[\s_\-]+", " ", str(s).lower()).strip()
_OVR = {_norm(k): v for k, v in CLASS_OVERRIDES.items()}

def classify_name(name):
    # -> a shape name, '__generic__', '__negative__', '__drop_image__' or '__unknown__'
    n = _norm(name)
    if n in _OVR:
        v = _OVR[n]
        return "__generic__" if v == "microplastic" else v
    if any(k in n for k in NEGATIVE_KEYWORDS): return "__negative__"
    hits = [s for s, kws in SHAPE_KEYWORDS.items() if any(k in n for k in kws)]
    if len(hits) == 1: return hits[0]
    if len(hits) > 1 or n in GENERIC_EXACT or "plastic" in n: return "__generic__"
    return "__unknown__"

def load_names(dest):
    names = yaml.safe_load(open(Path(dest, "data.yaml"))).get("names", [])
    if isinstance(names, dict): names = [names[k] for k in sorted(names, key=int)]
    return [str(x) for x in names]

rows = []
for sid, dest in SOURCES:
    for i, n in enumerate(load_names(dest)):
        rows.append(dict(source=sid, class_id=i, original_name=n, mapped_to=classify_name(n)))
class_table = pd.DataFrame(rows)
display(class_table)
unknown = class_table[class_table.mapped_to == "__unknown__"]
if len(unknown):
    print(f"⚠ {len(unknown)} unrecognised class name(s) -> policy '{UNKNOWN_POLICY}'. Add them to CLASS_OVERRIDES if they are plastic.")

In [ ]:
def to_target(kind):
    # -> ('box', class_index) | ('negative', None) | ('drop', reason)
    if kind == "__unknown__":
        kind = "__negative__" if UNKNOWN_POLICY == "negative" else "__drop_image__"
    if kind == "__negative__":   return ("negative", None)
    if kind == "__drop_image__": return ("drop", "unknown/override class")
    if LABEL_MODE == "single":   return ("box", 0)
    if kind == "__generic__":    return ("drop", "generic label in shape mode")
    return ("box", SHAPE_CLASSES.index(kind))

def iter_image_label_pairs(dest):
    for img_dir in sorted(p for p in Path(dest).rglob("images") if p.is_dir()):
        lbl_dir = img_dir.parent / "labels"
        for p in sorted(img_dir.iterdir()):
            if p.suffix.lower() in IMG_EXT:
                yield p, lbl_dir / f"{p.stem}.txt"

def parse_line(line):
    parts = line.split()
    if len(parts) < 5: return None
    try: cls, vals = int(float(parts[0])), [float(v) for v in parts[1:]]
    except ValueError: return None
    if len(vals) == 4:
        x, y, w, h = vals
    elif len(vals) >= 6 and len(vals) % 2 == 0:            # segmentation polygon -> bounding box
        xs, ys = vals[0::2], vals[1::2]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        x, y, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
    else:
        return None
    x1, y1 = max(0.0, x - w / 2), max(0.0, y - h / 2)       # clip to image
    x2, y2 = min(1.0, x + w / 2), min(1.0, y + h / 2)
    if x2 - x1 <= 1e-4 or y2 - y1 <= 1e-4: return None
    return cls, ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)

records, stats = [], defaultdict(Counter)
for si, (sid, dest) in enumerate(SOURCES):
    targets = [to_target(classify_name(n)) for n in load_names(dest)]
    for img, lbl in iter_image_label_pairs(dest):
        st = stats[sid]; st["images"] += 1
        boxes, drop = set(), None
        lines = lbl.read_text().splitlines() if lbl.exists() else []
        for line in lines:
            parsed = parse_line(line)
            if parsed is None:
                if line.strip(): st["bad label lines"] += 1
                continue
            cls, b = parsed
            if cls >= len(targets): st["bad label lines"] += 1; continue
            kind, val = targets[cls]
            if kind == "drop": drop = val; break
            if kind == "negative": st["negative boxes removed"] += 1; continue
            boxes.add((val, *(round(v, 6) for v in b)))
        if drop:
            st[f"dropped: {drop}"] += 1; continue
        st["kept"] += 1; st["boxes"] += len(boxes)
        if not boxes: st["background (no boxes)"] += 1
        records.append(dict(path=str(img), source=sid, src_idx=si, boxes=sorted(boxes),
                            stem_key=img.stem.split(".rf.")[0]))

CLASS_NAMES = ["microplastic"] if LABEL_MODE == "single" else SHAPE_CLASSES
if LABEL_MODE == "shape":   # drop shape classes that no dataset provides, re-index the rest
    used = sorted({b[0] for r in records for b in r["boxes"]})
    remap = {old: new for new, old in enumerate(used)}
    CLASS_NAMES = [CLASS_NAMES[i] for i in used]
    for r in records: r["boxes"] = [(remap[b[0]], *b[1:]) for b in r["boxes"]]

source_stats = pd.DataFrame(stats).T.fillna(0).astype(int)
display(source_stats)
print(f"Classes: {CLASS_NAMES} | images kept: {len(records)} | boxes: {sum(len(r['boxes']) for r in records)}")

## 4. Find duplicates across all datasets (prevents train/test leakage)
Two images are put in the same group if they share a Roboflow filename stem (`name_jpg.rf.<hash>` → augmented copies)
**or** one is a near-copy of the other under any flip/90° rotation (perceptual hash, then confirmed by high-pass pixel correlation so
that different microscope images with similar backgrounds are not merged).

In [ ]:
def load_thumb(path):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return None if im is None else cv2.resize(im, (64, 64), interpolation=cv2.INTER_AREA)

def phash(t):
    a = cv2.resize(t, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(a)[:8, :8].ravel()
    return np.packbits(d > np.median(d[1:])).view(np.uint64)[0]

def highpass_vec(t):
    f = t.astype(np.float32)
    h = f - cv2.GaussianBlur(f, (0, 0), 3)
    h -= h.mean(); n = np.linalg.norm(h)
    return (h / n if n > 1e-6 else h).ravel()

def dihedral(t):
    return [np.ascontiguousarray(np.rot90(b, k)) for b in (t, np.fliplr(t)) for k in range(4)]

if hasattr(np, "bitwise_count"):
    popcount = lambda x: np.bitwise_count(x)
else:
    _LUT = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    popcount = lambda x: _LUT[np.ascontiguousarray(x).view(np.uint8)].reshape(*x.shape, 8).sum(-1)

with ThreadPoolExecutor(8) as ex:
    thumbs = list(tqdm(ex.map(load_thumb, [r["path"] for r in records]), total=len(records), desc="reading"))
bad = [i for i, t in enumerate(thumbs) if t is None]
if bad: print(f"Removing {len(bad)} unreadable images")
records = [r for r, t in zip(records, thumbs) if t is not None]
thumbs = [t for t in thumbs if t is not None]
N = len(records)

parent = list(range(N))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[max(ra, rb)] = min(ra, rb)

# (a) same Roboflow stem within one dataset -> augmented copies of one original
#     (not across datasets: generic names like "1_jpg" or "IMG_0001_JPG" would merge unrelated images)
first = {}
for i, r in enumerate(records):
    union(i, first.setdefault((r["source"], r["stem_key"]), i))

# (b) visual near-duplicates, invariant to flips and 90° rotations
H = np.array([phash(t) for t in thumbs], dtype=np.uint64)
HP = np.stack([highpass_vec(t) for t in thumbs]).astype(np.float16)
visual_pairs = cross_source_pairs = 0; CROSS_EXAMPLES = []
for i in tqdm(range(N - 1), desc="matching"):
    variants = dihedral(thumbs[i])
    vh = np.array([phash(v) for v in variants], dtype=np.uint64)
    d = np.stack([popcount(H[i + 1:] ^ h) for h in vh])          # 8 x (N-i-1)
    dmin, vbest = d.min(0), d.argmin(0)
    cand = np.nonzero(dmin <= PHASH_MAX_DIST)[0]
    if cand.size == 0: continue
    if cand.size > 300: cand = cand[np.argsort(dmin[cand])[:300]]
    vhp = np.stack([highpass_vec(v) for v in variants])
    corr = (HP[i + 1 + cand].astype(np.float32) * vhp[vbest[cand]]).sum(1)
    for j in (i + 1 + cand[corr >= DUP_CORR]):
        visual_pairs += 1
        if records[i]["source"] != records[j]["source"]:
            cross_source_pairs += 1; CROSS_EXAMPLES.append((records[i]["path"], records[int(j)]["path"]))
        union(i, int(j))

groups = defaultdict(list)
for i in range(N): groups[find(i)].append(i)
sizes = sorted((len(g) for g in groups.values()), reverse=True)
print(f"{N} images -> {len(groups)} unique groups | visual duplicate pairs: {visual_pairs} "
      f"(across different datasets: {cross_source_pairs}) | largest groups: {sizes[:5]}")
if sizes[0] > 0.05 * N:
    print("⚠ One group holds >5% of all images - duplicate thresholds may be too loose (raise DUP_CORR).")

## 5. Split, add underwater-style training copies, write the merged dataset

In [ ]:
rng = random.Random(SEED)

# group duplicate-groups by source (source of the group's first image), split per source
by_source = defaultdict(list)
for root, members in groups.items():
    by_source[records[min(members)]["source"]].append(sorted(members))

assign = {}  # record index -> split
for sid, glist in by_source.items():
    rng.shuffle(glist)
    if MAX_IMAGES_PER_SOURCE:
        kept, n = [], 0
        for g in glist:
            if n >= MAX_IMAGES_PER_SOURCE: break
            kept.append(g); n += len(g)
        glist = kept
    ng = len(glist)
    n_test = max(1, round(SPLIT[2] * ng)) if ng >= 10 else 0
    n_val  = max(1, round(SPLIT[1] * ng)) if ng >= 10 else 0
    for k, g in enumerate(glist):
        split = "test" if k < n_test else "val" if k < n_test + n_val else "train"
        if split == "train":
            for i in g: assign[i] = "train"          # augmented copies are fine for training
        else:
            assign[g[0]] = split                       # one representative per group in val/test

def underwater(img, r):
    # Physically-motivated underwater look: red attenuates fastest, blue-green veiling light, blur, sensor noise.
    x = img.astype(np.float32) / 255.0
    if x.ndim == 2: x = np.repeat(x[..., None], 3, -1)
    depth = r.uniform(0.3, 1.0)
    x *= np.array([1 - 0.15 * depth, 1 - 0.05 * depth, 1 - 0.6 * depth], np.float32)       # B, G, R
    veil = np.array([r.uniform(0.35, 0.6), r.uniform(0.3, 0.55), r.uniform(0.03, 0.2)], np.float32)
    t = r.uniform(0.6, 0.9)
    x = x * t + veil * (1 - t)
    if r.random() < 0.6:
        k = r.choice([3, 5]); x = cv2.GaussianBlur(x, (k, k), 0)
    x += np.random.default_rng(r.randrange(1 << 30)).normal(0, r.uniform(0.005, 0.025), x.shape).astype(np.float32)
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)

shutil.rmtree(MERGED, ignore_errors=True)
for s in ("train", "val", "test"):
    os.makedirs(f"{MERGED}/{s}/images"); os.makedirs(f"{MERGED}/{s}/labels")
os.makedirs(f"{MERGED}/lists")

def label_text(boxes): return "".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in boxes)

manifest, counter = [], Counter()
for i, split in sorted(assign.items()):
    r = records[i]; counter[(split, r["src_idx"])] += 1
    name = f"s{r['src_idx']:02d}_{i:06d}"
    dst = Path(MERGED, split, "images", name + Path(r["path"]).suffix.lower())
    os.symlink(os.path.abspath(r["path"]), dst)
    Path(MERGED, split, "labels", name + ".txt").write_text(label_text(r["boxes"]))
    manifest.append(dict(image=str(dst), original=r["path"], source=r["source"], split=split,
                         group=find(i), boxes=len(r["boxes"]), underwater_copy=False))

uw_rng = random.Random(SEED + 1)
train_items = [m for m in manifest if m["split"] == "train"]
uw_pick = uw_rng.sample(train_items, int(UNDERWATER_AUG_FRACTION * len(train_items)))
for m in tqdm(uw_pick, desc="underwater copies"):
    img = cv2.imread(m["original"], cv2.IMREAD_UNCHANGED)
    if img is None: continue
    if img.ndim == 3 and img.shape[2] == 4: img = img[..., :3]
    stem = Path(m["image"]).stem + "_uw"
    dst = Path(MERGED, "train", "images", stem + ".jpg")
    cv2.imwrite(str(dst), underwater(img, uw_rng), [cv2.IMWRITE_JPEG_QUALITY, 92])
    shutil.copy(Path(MERGED, "train", "labels", Path(m["image"]).stem + ".txt"), Path(MERGED, "train", "labels", stem + ".txt"))
    manifest.append({**m, "image": str(dst), "underwater_copy": True})

manifest = pd.DataFrame(manifest)
manifest.to_csv(f"{OUT}/merged_manifest.csv", index=False)

DATA_YAML = f"{MERGED}/data.yaml"
yaml.safe_dump(dict(path=MERGED, train="train/images", val="val/images", test="test/images",
                    names={i: n for i, n in enumerate(CLASS_NAMES)}), open(DATA_YAML, "w"), sort_keys=False)

# one test list per source -> per-source evaluation later
SOURCE_YAMLS = {}
for sid, part in manifest[manifest.split == "test"].groupby("source"):
    slug = re.sub(r"[^a-z0-9]+", "_", sid.lower())
    lst = f"{MERGED}/lists/test_{slug}.txt"
    Path(lst).write_text("\n".join(part.image) + "\n")
    y = f"{MERGED}/lists/test_{slug}.yaml"
    yaml.safe_dump(dict(path=MERGED, train="train/images", val=lst, test=lst,
                        names={i: n for i, n in enumerate(CLASS_NAMES)}), open(y, "w"), sort_keys=False)
    SOURCE_YAMLS[sid] = (y, len(part))

summary = manifest.pivot_table(index="source", columns="split", values="image", aggfunc="count", fill_value=0)
summary = summary.reindex(columns=[c for c in ["train", "val", "test"] if c in summary.columns])
summary.loc["TOTAL"] = summary.sum()
display(summary)
print(f"(train includes {int(manifest.underwater_copy.sum())} underwater-style copies)  ->  {DATA_YAML}")

### Sanity check — labels and object sizes

In [ ]:
def read_boxes(img_path):
    lp = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    return [list(map(float, l.split())) for l in lp.read_text().splitlines() if l.strip()] if lp.exists() else []

sample = manifest[(manifest.split == "train") & (manifest.boxes > 0)].sample(
    min(12, int(((manifest.split == "train") & (manifest.boxes > 0)).sum())), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for ax in axes.ravel(): ax.axis("off")
for ax, (_, m) in zip(axes.ravel(), sample.iterrows()):
    im = cv2.cvtColor(cv2.imread(m.image), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for c, x, y, bw, bh in read_boxes(m.image):
        cv2.rectangle(im, (int((x - bw / 2) * w), int((y - bh / 2) * h)), (int((x + bw / 2) * w), int((y + bh / 2) * h)),
                      (255, 40, 40), max(2, w // 300))
    ax.imshow(im); ax.set_title(f"{m.source.split('/')[-1]}{' (uw)' if m.underwater_copy else ''}", fontsize=9)
plt.tight_layout(); plt.show()

# object size in pixels *at training resolution* — very small objects need a high IMGSZ
sizes = np.array([np.sqrt(b[3] * b[4]) * IMGSZ for p in manifest.image[manifest.split == "train"] for b in read_boxes(p)])
if len(sizes):
    plt.figure(figsize=(8, 3)); plt.hist(np.clip(sizes, 0, 200), bins=60)
    plt.xlabel(f"object size (px) at IMGSZ={IMGSZ}"); plt.ylabel("count"); plt.title("Training object sizes"); plt.show()
    print(f"median {np.median(sizes):.0f}px | {100 * np.mean(sizes < 16):.1f}% of objects are under 16px at this resolution")

## 6. Train

In [ ]:
from ultralytics import YOLO

if RESUME_FROM:
    # Continue an interrupted run from its last saved epoch (same optimizer state, LR schedule and epoch count).
    # The dataset cells above must have been re-run first: they rebuild the identical split (same SEED).
    ckpt = Path(OUT, "runs", "train", "weights", "last.pt")
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    if Path(RESUME_FROM).resolve() != ckpt.resolve():
        shutil.copy(RESUME_FROM, ckpt)
    model = YOLO(str(ckpt))
    model.train(resume=True, device=DEVICE, batch=BATCH)
else:
    model = YOLO(MODEL)
    model.train(
        data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, time=TIME_HOURS, patience=PATIENCE, batch=BATCH, device=DEVICE,
        workers=WORKERS, project=f"{OUT}/runs", name="train", exist_ok=True, seed=SEED,
        cos_lr=True, close_mosaic=15,
        flipud=0.5, fliplr=0.5,      # particles have no "up" — vertical flips are valid
        mixup=0.1, plots=True,
    )
BEST = Path(model.trainer.best)
shutil.copy(BEST, f"{OUT}/best.pt")
print("Best weights:", BEST)

## 7. Evaluate on the held-out test set
- **Overall** — the headline number
- **Per class** — which particle types are hard
- **Per source** — if one dataset scores far lower, the model is relying on that dataset's look rather than on the particles
- **At 640 px** — what to expect after shrinking the input for the Raspberry Pi

In [ ]:
best = YOLO(str(BEST))
def metrics(m):
    return dict(precision=round(float(m.box.mp), 4), recall=round(float(m.box.mr), 4),
                mAP50=round(float(m.box.map50), 4), **{"mAP50-95": round(float(m.box.map), 4)})

m_test = best.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                  project=f"{OUT}/runs", name="test_eval", exist_ok=True, plots=True)
results = {"overall_test": metrics(m_test), "imgsz": IMGSZ, "model": MODEL, "label_mode": LABEL_MODE,
           "classes": CLASS_NAMES, "split_counts": summary.to_dict()}

per_class = []
for k, c in enumerate(m_test.box.ap_class_index):
    p, r, ap50, ap = m_test.box.class_result(k)
    per_class.append(dict(cls=CLASS_NAMES[int(c)], precision=p, recall=r, mAP50=ap50, **{"mAP50-95": ap}))
per_class = pd.DataFrame(per_class).round(4); results["per_class"] = per_class.to_dict("records")

per_source = []
for sid, (y, n) in SOURCE_YAMLS.items():
    m = best.val(data=y, split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=False, verbose=False,
                 project=f"{OUT}/runs", name="per_source", exist_ok=True)
    per_source.append(dict(source=sid, test_images=n, **metrics(m)))
per_source = pd.DataFrame(per_source); results["per_source"] = per_source.to_dict("records")

if IMGSZ != 640:
    m640 = best.val(data=DATA_YAML, split="test", imgsz=640, batch=BATCH, device=DEVICE, plots=False, verbose=False,
                    project=f"{OUT}/runs", name="test_640", exist_ok=True)
    results["overall_test_640px"] = metrics(m640)

print("TEST SET (overall):", results["overall_test"])
if "overall_test_640px" in results: print("TEST SET at 640px :", results["overall_test_640px"])
display(per_class); display(per_source)
json.dump(results, open(f"{OUT}/results_summary.json", "w"), indent=2, default=float)

In [ ]:
# Predictions on random test images (confidence 0.25)
test_imgs = manifest[manifest.split == "test"].sample(min(8, int((manifest.split == "test").sum())), random_state=SEED).image.tolist()
preds = best.predict(test_imgs, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax in axes.ravel(): ax.axis("off")
for ax, p, path in zip(axes.ravel(), preds, test_imgs):
    ax.imshow(cv2.cvtColor(p.plot(line_width=2), cv2.COLOR_BGR2RGB))
    ax.set_title(f"pred {len(p.boxes)} | true {len(read_boxes(path))}", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/test_predictions.png", dpi=110); plt.show()

## 8. Export for the Raspberry Pi 4 (ONNX + NCNN)

In [ ]:
EXPORT = Path(OUT, "export"); EXPORT.mkdir(exist_ok=True)
for fmt, sz in [("onnx", 640), ("onnx", IMGSZ), ("ncnn", 640)]:
    try:
        out = Path(YOLO(str(BEST)).export(format=fmt, imgsz=sz, verbose=False))
        dst = EXPORT / (f"best_{sz}.onnx" if fmt == "onnx" else f"best_ncnn_{sz}")
        if dst.exists(): shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
        shutil.move(str(out), dst); print(f"✓ {fmt} @ {sz}px -> {dst}")
    except Exception as e:
        print(f"✗ {fmt} @ {sz}px: {type(e).__name__}: {e}")

## 9. (Optional) Generalisation check — hold out one whole dataset
Trains a second model **without** one source dataset (and without any duplicate of its images from other datasets), then tests on
that unseen dataset. This number will be lower than the headline score — it is the more realistic preview of what happens with a
new camera. Takes roughly half the main training time.

In [ ]:
RUN_HOLDOUT = False
HOLDOUT_SOURCE = None        # e.g. "university-of-alabama-zwtwm/microplastic-detection"; None = largest source
HOLDOUT_EPOCHS = 60

if RUN_HOLDOUT:
    real = manifest[~manifest.underwater_copy]
    hs = HOLDOUT_SOURCE or real.source.value_counts().index[0]
    tainted = set(real.group[real.source == hs])                  # groups touching the held-out dataset
    train_imgs = manifest[(manifest.split != "test") & ~manifest.group.isin(tainted)]
    test_imgs_h = real[(real.source == hs)].drop_duplicates("group")
    lst = lambda name, df: (Path(MERGED, "lists", name).write_text("\n".join(df.image) + "\n"), str(Path(MERGED, "lists", name)))[1]
    tr = lst("holdout_train.txt", train_imgs[train_imgs.split == "train"])
    va = lst("holdout_val.txt", train_imgs[train_imgs.split == "val"])
    te = lst("holdout_test.txt", test_imgs_h)
    hy = f"{MERGED}/lists/holdout.yaml"
    yaml.safe_dump(dict(path=MERGED, train=tr, val=va, test=te, names={i: n for i, n in enumerate(CLASS_NAMES)}), open(hy, "w"))
    print(f"Held out: {hs} ({len(test_imgs_h)} unique images) | training on {len(train_imgs[train_imgs.split == 'train'])} images")
    hm = YOLO(MODEL)
    hm.train(data=hy, imgsz=IMGSZ, epochs=HOLDOUT_EPOCHS, patience=PATIENCE, batch=BATCH, device=DEVICE, workers=WORKERS,
             project=f"{OUT}/runs", name="holdout", exist_ok=True, seed=SEED, cos_lr=True, flipud=0.5, plots=False)
    mh = YOLO(hm.trainer.best).val(data=hy, split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=False,
                                    project=f"{OUT}/runs", name="holdout_eval", exist_ok=True)
    results["holdout"] = {"source": hs, **metrics(mh)}
    print("Unseen-dataset result:", results["holdout"])
    json.dump(results, open(f"{OUT}/results_summary.json", "w"), indent=2, default=float)

## 10. Package results

In [ ]:
zip_path = Path(OUT, "microplastic_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in [Path(OUT, "best.pt"), Path(OUT, "results_summary.json"), Path(OUT, "merged_manifest.csv"), Path(OUT, "test_predictions.png")]:
        if p.exists(): z.write(p, p.name)
    for p in Path(OUT, "export").rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(OUT))
    for run in ["train", "test_eval"]:
        for p in Path(OUT, "runs", run).glob("*"):
            if p.is_file() and p.suffix in {".png", ".jpg", ".csv", ".yaml"}: z.write(p, f"{run}/{p.name}")
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB) — download it from the Output panel")

## Notes for the review
- Quote the result as: *"precision P / recall R / mAP50 M on a held-out test set of N images merged from K public microplastic datasets, with cross-dataset duplicates removed."*
- **Recall** is the "how many particles did we miss" number — if the requirement is "detect every microplastic", lead with it, and lower the confidence threshold on the Pi to trade precision for recall.
- Show the per-source table: consistent scores across datasets is evidence the model learned particles, not one lab's background.
- Check each dataset's licence on its Universe page (most are CC BY 4.0 → cite them in the report).
- Next step once a camera is shortlisted: photograph real fragments in a water tank with that camera, label ~200 images, add them as another source, and fine-tune from `best.pt`.